# 03 - Compare runs

Notebook para comparar runs guardados en `mlruns/`:

- métricas clave
- configuración resumida
- curvas de equity / benchmark cuando existan
- series de IC cuando existan

Está pensado para seguir funcionando a medida que vayas generando más experimentos.

In [ ]:
from pathlib import Path
import sys
import pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

plt.style.use('seaborn-v0_8')
ROOT


In [ ]:
from qlib_project.bootstrap import init_qlib

init_qlib()


In [ ]:
MLRUNS_ROOT = ROOT / 'mlruns'
MLRUNS_ROOT


In [ ]:
def read_metric_file(path: Path):
    rows = []
    if not path.exists():
        return rows
    for line in path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) >= 2:
            timestamp = int(parts[0])
            value = float(parts[1])
            step = int(parts[2]) if len(parts) >= 3 else None
            rows.append({'timestamp': timestamp, 'value': value, 'step': step})
    return rows


def read_param_file(path: Path):
    if not path.exists():
        return None
    return path.read_text().strip()


def maybe_load_pickle(path: Path):
    if not path.exists():
        return None
    with open(path, 'rb') as f:
        return pickle.load(f)


def collect_runs(mlruns_root: Path):
    runs = []
    for exp_dir in sorted(mlruns_root.iterdir()):
        if not exp_dir.is_dir() or exp_dir.name.startswith('.') or exp_dir.name == '.trash':
            continue
        for run_dir in sorted(exp_dir.iterdir()):
            if not run_dir.is_dir():
                continue
            if not (run_dir / 'meta.yaml').exists():
                continue

            meta = yaml.safe_load((run_dir / 'meta.yaml').read_text())
            metrics_dir = run_dir / 'metrics'
            params_dir = run_dir / 'params'
            artifacts_dir = run_dir / 'artifacts'

            latest_metrics = {}
            full_metrics = {}
            if metrics_dir.exists():
                for metric_file in metrics_dir.iterdir():
                    if metric_file.is_file():
                        rows = read_metric_file(metric_file)
                        full_metrics[metric_file.name] = rows
                        if rows:
                            latest_metrics[metric_file.name] = rows[-1]['value']

            params = {}
            if params_dir.exists():
                for param_file in params_dir.iterdir():
                    if param_file.is_file():
                        params[param_file.name] = read_param_file(param_file)

            runs.append({
                'experiment_id': exp_dir.name,
                'run_id': run_dir.name,
                'meta': meta,
                'metrics': latest_metrics,
                'metrics_full': full_metrics,
                'params': params,
                'artifacts_dir': artifacts_dir,
            })
    return runs


runs = collect_runs(MLRUNS_ROOT)
len(runs)


## Tabla resumen de runs

In [ ]:
def summarize_run(run):
    m = run['metrics']
    p = run['params']
    return {
        'experiment_id': run['experiment_id'],
        'run_id': run['run_id'],
        'model': p.get('model.class'),
        'handler': p.get('dataset.kwargs.handler.class'),
        'instruments': p.get('dataset.kwargs.handler.kwargs.instruments'),
        'IC': m.get('IC'),
        'ICIR': m.get('ICIR'),
        'Rank IC': m.get('Rank IC'),
        'Rank ICIR': m.get('Rank ICIR'),
        'ann_ret_cost': m.get('1day.excess_return_with_cost.annualized_return'),
        'ir_cost': m.get('1day.excess_return_with_cost.information_ratio'),
        'mdd_cost': m.get('1day.excess_return_with_cost.max_drawdown'),
        'mlflow_run_name': run['meta'].get('run_name'),
        'status': run['meta'].get('status'),
    }

summary_df = pd.DataFrame([summarize_run(r) for r in runs])
summary_df


## Ranking automático y mejor run

In [ ]:
ranking_df = summary_df.copy()

if not ranking_df.empty:
    ranking_df['mdd_abs'] = ranking_df['mdd_cost'].abs()
    ranking_df['rank_ir_cost'] = ranking_df['ir_cost'].rank(ascending=False, method='min')
    ranking_df['rank_ann_ret_cost'] = ranking_df['ann_ret_cost'].rank(ascending=False, method='min')
    ranking_df['rank_ic'] = ranking_df['IC'].rank(ascending=False, method='min')
    ranking_df['rank_mdd'] = ranking_df['mdd_abs'].rank(ascending=True, method='min')
    ranking_df['composite_score'] = (
        0.40 * ranking_df['rank_ir_cost'] +
        0.30 * ranking_df['rank_ann_ret_cost'] +
        0.20 * ranking_df['rank_ic'] +
        0.10 * ranking_df['rank_mdd']
    )
    ranking_df = ranking_df.sort_values(['composite_score', 'ir_cost', 'ann_ret_cost'], ascending=[True, False, False]).reset_index(drop=True)

ranking_df


In [ ]:
if not ranking_df.empty:
    best_run = ranking_df.iloc[0]
    best_run_summary = pd.DataFrame({
        'value': [
            best_run['run_id'],
            best_run['instruments'],
            best_run['handler'],
            best_run['IC'],
            best_run['Rank IC'],
            best_run['ann_ret_cost'],
            best_run['ir_cost'],
            best_run['mdd_cost'],
            best_run['composite_score'],
        ]
    }, index=[
        'run_id',
        'instruments',
        'handler',
        'IC',
        'Rank IC',
        'ann_ret_cost',
        'ir_cost',
        'mdd_cost',
        'composite_score',
    ])
    best_run_summary
else:
    'no runs found'


## Seleccionar runs a comparar

In [ ]:
selected_run_ids = ranking_df['run_id'].tolist() if not ranking_df.empty else []
selected_run_ids


In [ ]:
selected_runs = [r for r in runs if r['run_id'] in selected_run_ids]
len(selected_runs)


## Curvas de equity y benchmark

In [ ]:
def load_report(run):
    return maybe_load_pickle(run['artifacts_dir'] / 'portfolio_analysis' / 'report_normal_1day.pkl')

reports = {}
for run in selected_runs:
    report = load_report(run)
    if report is not None:
        report = report.copy()
        report['strategy_nav'] = (1 + report['return'].fillna(0)).cumprod()
        report['benchmark_nav'] = (1 + report['bench'].fillna(0)).cumprod()
        report['excess_return'] = report['return'].fillna(0) - report['bench'].fillna(0)
        report['excess_nav'] = (1 + report['excess_return']).cumprod()
        reports[run['run_id']] = report

list(reports.keys())


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)
for run_id, report in reports.items():
    label = run_id
    if not ranking_df.empty and run_id == ranking_df.iloc[0]['run_id']:
        label = f'{run_id}  ← best'
    report['strategy_nav'].plot(ax=axes[0], label=label)
    report['benchmark_nav'].plot(ax=axes[1], label=label)
    report['excess_nav'].plot(ax=axes[2], label=label)

axes[0].set_title('Strategy NAV')
axes[1].set_title('Benchmark NAV')
axes[2].set_title('Excess NAV')
for ax in axes:
    ax.legend()
plt.tight_layout()


## Drawdowns comparados

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for run_id, report in reports.items():
    label = run_id
    if not ranking_df.empty and run_id == ranking_df.iloc[0]['run_id']:
        label = f'{run_id}  ← best'
    dd = report['strategy_nav'] / report['strategy_nav'].cummax() - 1
    edd = report['excess_nav'] / report['excess_nav'].cummax() - 1
    dd.plot(ax=axes[0], label=label)
    edd.plot(ax=axes[1], label=label)

axes[0].set_title('Strategy drawdown')
axes[1].set_title('Excess drawdown')
for ax in axes:
    ax.legend()
plt.tight_layout()


## IC y Rank IC por run

In [ ]:
def load_series_artifact(run, relative_path):
    obj = maybe_load_pickle(run['artifacts_dir'] / relative_path)
    return obj if isinstance(obj, pd.Series) else None

ic_series = {}
ric_series = {}
for run in selected_runs:
    s1 = load_series_artifact(run, 'sig_analysis/ic.pkl')
    s2 = load_series_artifact(run, 'sig_analysis/ric.pkl')
    if s1 is not None:
        ic_series[run['run_id']] = s1
    if s2 is not None:
        ric_series[run['run_id']] = s2

list(ic_series.keys()), list(ric_series.keys())


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for run_id, s in ic_series.items():
    label = run_id
    if not ranking_df.empty and run_id == ranking_df.iloc[0]['run_id']:
        label = f'{run_id}  ← best'
    s.plot(ax=axes[0], label=label)
for run_id, s in ric_series.items():
    label = run_id
    if not ranking_df.empty and run_id == ranking_df.iloc[0]['run_id']:
        label = f'{run_id}  ← best'
    s.plot(ax=axes[1], label=label)

axes[0].axhline(0, color='black', linewidth=1)
axes[1].axhline(0, color='black', linewidth=1)
axes[0].set_title('Daily IC by run')
axes[1].set_title('Daily Rank IC by run')
for ax in axes:
    ax.legend()
plt.tight_layout()


## Curvas de entrenamiento LightGBM

In [ ]:
def metric_history_df(run, metric_name):
    rows = run['metrics_full'].get(metric_name, [])
    if not rows:
        return None
    return pd.DataFrame(rows)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for run in selected_runs:
    run_id = run['run_id']
    label = run_id
    if not ranking_df.empty and run_id == ranking_df.iloc[0]['run_id']:
        label = f'{run_id}  ← best'
    train_hist = metric_history_df(run, 'l2.train')
    valid_hist = metric_history_df(run, 'l2.valid')
    if train_hist is not None:
        axes[0].plot(train_hist['step'], train_hist['value'], label=label)
    if valid_hist is not None:
        axes[1].plot(valid_hist['step'], valid_hist['value'], label=label)

axes[0].set_title('l2.train')
axes[1].set_title('l2.valid')
axes[1].set_xlabel('Boosting round')
for ax in axes:
    ax.legend()
plt.tight_layout()


## Configuración resumida

In [ ]:
config_cols = [
    'model.class',
    'dataset.kwargs.handler.class',
    'dataset.kwargs.handler.kwargs.instruments',
    'model.kwargs.learning_rate',
    'model.kwargs.max_depth',
    'model.kwargs.num_leaves',
    'model.kwargs.subsample',
    'model.kwargs.colsample_bytree',
]
config_summary = []
for run in selected_runs:
    row = {'run_id': run['run_id']}
    for col in config_cols:
        row[col] = run['params'].get(col)
    config_summary.append(row)
pd.DataFrame(config_summary)


## Ideas para usar este notebook

- mantener un baseline fijo y comparar variaciones pequeñas
- mirar si mejora IR pero empeora drawdown
- evitar optimizar una sola métrica
- documentar cada run con cambios concretos en config
